# Week 4 — Hands-On: Machine Learning Basics

Topics: train/val/test split · feature scaling · cross-validation · evaluation metrics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report,
    roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay
)

# Reproducibility
np.random.seed(42)

## 1. Load Dataset

In [ ]:
cancer = load_breast_cancer(as_frame=True)
X, y = cancer.data, cancer.target
print(f'Features: {X.shape[1]},  Samples: {X.shape[0]}')
print(f'Class balance: {dict(y.value_counts())}')

## 2. Split & Scale

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape}, Test: {X_test_sc.shape}')

## 3. Cross-Validation

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train_sc, y_train, cv=cv, scoring='roc_auc')

print(f'CV ROC-AUC: {scores.mean():.4f} ± {scores.std():.4f}')
print(f'Per-fold  : {scores.round(4)}')

## 4. Final Evaluation on Test Set

In [ ]:
model.fit(X_train_sc, y_train)
y_pred  = model.predict(X_test_sc)
y_proba = model.predict_proba(X_test_sc)[:, 1]

print(f'Test Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'Test ROC-AUC  : {roc_auc_score(y_test, y_proba):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=cancer.target_names, ax=axes[0]
)
axes[0].set_title('Confusion Matrix')

RocCurveDisplay.from_estimator(model, X_test_sc, y_test, ax=axes[1])
axes[1].set_title('ROC Curve')

plt.tight_layout()
plt.show()

## 5. Exercises

1. Replace `LogisticRegression` with `sklearn.neighbors.KNeighborsClassifier`. How does ROC-AUC compare?
2. Try different values of `C` (regularisation strength) in `LogisticRegression`. Plot ROC-AUC vs `C`.
3. Apply `MinMaxScaler` instead of `StandardScaler`. Does it affect performance?
4. *Bonus*: use `sklearn.model_selection.learning_curve` to diagnose bias vs variance.